# THOR — Weighted multi-broker fusion: `FusionService` and `Hunter.search_all()`

<div align="center">
<img src="./figures/logo.svg" width="600">
</div>

**Author:** Fabio Ragosta (they/them), fix-term researcher at the University of Naples "Federico II"
**Tutorial 3 of 3.**

As shown in tutorial 2, `Hunter.search()`/`Hunter.cone_search()` only ever query **one broker at a time**
(the one loaded with `load()`). `FusionService` groups candidates by `object_id` and combines them with
per-broker weights — but if the input only ever comes from one broker, that fusion is a no-op in disguise.

`Hunter.search_all()` queries several brokers and fuses the results **before** running the rest of the
pipeline, so that an object seen by multiple brokers is genuinely cross-matched and classified with the
right weights. This notebook covers: the weighted-fusion formula, `search_all()` in action, the newly
implemented broker-agreement score, and a few known limitations.


## 1. `FusionService`: weights and formula

For every transient class, the fused classification is a **weighted average** of the probabilities
reported by the brokers that proposed that class:

```
fused[class] = Σ (broker_weight * broker_prob[class]) / Σ broker_weight
```

where the sum only runs over the brokers that actually reported that class (not over every broker
that was queried).


In [1]:
from thor.services.fusion import FusionService

fusion = FusionService()
print("Default weights:", fusion.weights)


Default weights: {'Fink': 0.38, 'ALeRCE': 0.67, 'Lasair': -0.05}


Let's verify the formula by hand, replicating `FusionService._combine()` on three hypothetical
classifications of the same object for "SN Ia":


In [2]:
probs = {"Fink": 0.60, "ALeRCE": 0.90, "Lasair": 0.50}
weights = fusion.weights

numerator = sum(weights[b] * p for b, p in probs.items())
denominator = sum(weights[b] for b in probs)

print("Manual computation:", numerator / denominator)


Manual computation: 0.806


## 2. `Hunter.search_all()` in action

As in tutorial 2, to keep this notebook runnable without network access we replace
`BrokerManager.load()` with simulated brokers. In production you can call `hunter.search_all([...])`
directly: each broker will make a real REST request (Fink, Lasair, ALeRCE).

We simulate the **same object** (`object_id="OBJ1"`) reported by three brokers with different
classifications, to see the weighted fusion actually working.


In [3]:
from unittest.mock import MagicMock
from thor.model import Candidate, Coordinates, BrokerInfo, Classification
from thor.hunter import Hunter

PROBS_PER_BROKER = {
    "fink":   {"name": "Fink",   "probabilities": {"SN Ia": 0.60, "SN II": 0.40}},
    "alerce": {"name": "ALeRCE", "probabilities": {"SN Ia": 0.90, "SN II": 0.10}},
    "lasair": {"name": "Lasair", "probabilities": {"SN Ia": 0.20, "SN II": 0.80}},
}

def fake_load(name, survey="ztf"):
    """Replaces BrokerManager.load(): in production instantiates a real REST client per broker."""
    info = PROBS_PER_BROKER[name]
    broker = MagicMock()
    broker.name = info["name"]

    c = Candidate(coordinates=Coordinates(ra=150.324, dec=2.211))
    c.add_broker(BrokerInfo(broker=info["name"], object_id="OBJ1"))
    c.classification = Classification(probabilities=info["probabilities"])

    broker.search.return_value = [c]
    return broker

hunter = Hunter()
hunter.manager.load = fake_load  # tutorial-only override

fused = hunter.search_all(["fink", "alerce", "lasair"], classifier="SN")

print("Number of fused candidates:", len(fused))
for c in fused:
    print(c)
    print("  fused classification:", c.classification.probabilities)
    print("  contributing brokers:", list(c.broker_classifications.keys()))
    print("  ranking.agreement:", round(c.ranking.agreement, 3))
    print("  ranking.total:", c.ranking.total)


Number of fused candidates: 1
Candidate(object='OBJ1', class='SN Ia (82.10%)', RA=150.32400, Dec=2.21100, brokers=[Fink])
  fused classification: {'SN Ia': 0.8210000000000001, 'SN II': 0.17900000000000002}
  contributing brokers: ['Fink', 'ALeRCE', 'Lasair']
  ranking.agreement: 0.659
  ranking.total: 0.6134


Let's sort by total score, the way one would when triaging a real alert list:


In [4]:
fused_sorted = sorted(fused, key=lambda c: c.score, reverse=True)
for c in fused_sorted:
    print(f"{c.broker_info[0].object_id}: score={c.score:.3f}  "
          f"class={c.classification.best_class} ({c.classification.best_probability:.0%})")


OBJ1: score=0.613  class=SN Ia (82%)


### `broker_kwargs`: per-broker parameters

If one broker needs a different parameter from the others (e.g. a `classifier` mapped differently for
the LSST survey, as seen in tutorial 2 for Fink), `search_all()` lets you specify it per broker with
`broker_kwargs`, while `**kwargs` stays the shared default.


In [5]:
fused = hunter.search_all(
    ["fink", "alerce", "lasair"],
    classifier="SN",                                            # used by alerce and lasair
    broker_kwargs={"fink": {"classifier": "most_likely_sn"}},   # override for fink only
)
print("OK, ran with per-broker parameters.")


OK, ran with per-broker parameters.


## 3. Broker agreement: how it's actually computed

`RankingService.agreement_score()` used to be a fixed placeholder (always `0.5`). It now computes the
**mean pairwise cosine similarity** between the probability distributions reported by every broker that
classified the candidate:

```
agreement = mean over broker pairs (i, j) of cosine(prob_i, prob_j)
```

Cosine similarity is used instead of a strict "do the top classes match" check because it degrades
gracefully: two brokers that split their probability mass between the same two classes still score as
"agreeing" even if their nominal top class differs by a hair, while brokers pointing at entirely
different classes score close to `0`. With 0 or 1 broker there is nothing to compare, so the score falls
back to the neutral `0.5` seen in tutorial 1.


In [6]:
from thor.services.ranking import RankingService

ranking = RankingService()

# Two brokers in full agreement
c = Candidate()
c.broker_classifications = {
    "Fink": Classification(probabilities={"SN Ia": 0.9, "SN II": 0.1}),
    "ALeRCE": Classification(probabilities={"SN Ia": 0.9, "SN II": 0.1}),
}
print("Two brokers, identical:", ranking.agreement_score(c))

# Two brokers in total disagreement (disjoint classes)
c = Candidate()
c.broker_classifications = {
    "Fink": Classification(probabilities={"SN Ia": 1.0}),
    "Lasair": Classification(probabilities={"AGN": 1.0}),
}
print("Two brokers, total disagreement:", ranking.agreement_score(c))

# The three brokers used above in search_all()
c = Candidate()
c.broker_classifications = {
    "Fink": Classification(probabilities={"SN Ia": 0.60, "SN II": 0.40}),
    "ALeRCE": Classification(probabilities={"SN Ia": 0.90, "SN II": 0.10}),
    "Lasair": Classification(probabilities={"SN Ia": 0.20, "SN II": 0.80}),
}
print("Three brokers, partial disagreement:", ranking.agreement_score(c))


Two brokers, identical: 0.9999999999999999
Two brokers, total disagreement: 0.0
Three brokers, partial disagreement: 0.6587814153851731


Note the design choice: brokers that didn't classify at all (empty probabilities) are excluded from
the comparison rather than automatically counted as "disagreeing" — this seemed the more defensible
default, but it is a design decision rather than the only reasonable one.


## 4. Known limitations

- **Negative weights and per-class normalization.** Lasair's default weight is `-0.05`. The
  normalization in `_combine()` is done *per class*, not globally: if the weights of the brokers that
  proposed a given class sum to exactly zero for that class, you get a `ZeroDivisionError`. Reproduced
  below with two brokers whose weights cancel each other out:


In [7]:
fusion_edge_case = FusionService(weights={"Lasair": -0.05, "CustomBroker": 0.05})

c1 = Candidate()
c1.add_broker(BrokerInfo(broker="Lasair", object_id="OBJ3"))
c1.classification = Classification(probabilities={"AGN": 0.7})

c2 = Candidate()
c2.add_broker(BrokerInfo(broker="CustomBroker", object_id="OBJ3"))
c2.classification = Classification(probabilities={"AGN": 0.5})

try:
    fusion_edge_case.run([c1, c2])
except ZeroDivisionError as exc:
    print("Reproduced bug in FusionService._combine():", repr(exc))


Reproduced bug in FusionService._combine(): ZeroDivisionError('float division by zero')


- **`Hunter.cone_search()`** has an unrelated bug (`self.fusion(candidates)` instead of
  `self.fusion.run(candidates)`) — seen in tutorial 2. `search_all()` is unaffected, since it always
  calls `self.fusion.run(...)` correctly.
- **`CalibrationService`** still doesn't apply any real statistical calibration (isotonic/Platt/beta
  scaling are planned future work).
- **`agreement_score`** is now implemented (section 3 above) — no longer on this list.

These limitations are documented in the THOR paper as ongoing validation work; here we've simply made
them visible and reproducible, so it's clear exactly where to focus before running this at scale on
real data.
